In [2]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import librosa
from tqdm import tqdm

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").cuda()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
import os

base_path = ".."   # go up one folder

audio_path = os.path.join(base_path, sample["audio"])

In [10]:
import os
import json
import librosa
import pandas as pd
from tqdm import tqdm
import torch

# Load dataset.jsonl
data = []
with open("../dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

# Create mapping filename -> reference text
audio_to_text = {}
for sample in data:
    filename = sample["audio"].split("\\")[-1]   # get 825780_0.wav
    audio_to_text[filename] = sample["text"]

results = []

segment_folder = "../segments"
segment_files = os.listdir(segment_folder)

for i, filename in enumerate(tqdm(segment_files)):
    if i >= 1000:
        break

    if filename not in audio_to_text:
        continue

    audio_path = os.path.join(segment_folder, filename)
    reference_text = audio_to_text[filename]

    # Load audio
    speech, sr = librosa.load(audio_path, sr=16000)

    inputs = processor.feature_extractor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.cuda()

    with torch.no_grad():
        predicted_ids = model.generate(
            inputs,
            max_new_tokens=128,
            num_beams=1,
            no_repeat_ngram_size=2,
            repetition_penalty=1.2
        )

    raw_text = processor.tokenizer.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    results.append({
        "audio": filename,
        "raw_asr": raw_text,
        "reference": reference_text
    })

    if i % 100 == 0:
        pd.DataFrame(results).to_csv(
            "C:/Josh_assin/temp-eval/raw_asr_vs_reference1.csv",
            index=False
        )

# Final save
pd.DataFrame(results).to_csv(
    "C:/Josh_assin/temp-eval/raw_asr_vs_reference1.csv",
    index=False
)

print("Done")

 17%|█▋        | 1000/5941 [11:35<57:18,  1.44it/s] 

Done
